# 🌪️ CATALYST RISK: Mathematical Dynamics & Deep Dive
## Advanced Stochastic Catastrophe Modelling

This interactive notebook demonstrates the core mathematical and statistical dynamics powering the CATALYST RISK engine. 
In real-world (tier-1) reinsurance and quantitative risk models (like RMS or AIR), a catastrophe model is broken down into four independent, highly complex modules:
1. **Stochastic Hazard**: Simulating tens of thousands of years of extreme natural events (long-tail distributions).
2. **Geospatial Exposure**: The physical properties mapping.
3. **Vulnerability**: Non-linear damage functions transforming physical intensity into a structural damage ratio.
4. **Financial Intelligence**: The application of complex, non-linear insurance structures (retentions, limits, co-participation).

---

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set dark theme to match CATALYST UI
pio.templates.default = "plotly_dark"

### 1. Stochastic Hazard Dynamics (Log-Normal Distributions)
Catastrophic hazard intensities (e.g., wind speeds, flood depths, seismic Peak Ground Acceleration (PGA)) are strictly non-negative and highly skewed. They are best modeled using heavy-tailed distributions like the **Log-Normal** or **Extreme Value (Gumbel/Frechet/Weibull)** distributions.

In the CATALYST engine, hazard intensity $I$ is sampled from a Log-Normal distribution:
$$ f(I; \mu, \sigma) = \frac{1}{I\sigma\sqrt{2\pi}} e^{-\frac{(\ln I - \mu)^2}{2\sigma^2}} $$

Where $\mu$ and $\sigma$ dictate the mean expected severity and the "fatness" of the extreme tail.

In [ ]:
def plot_hazard_distributions():
    np.random.seed(42)
    sims = 100000
    
    # Different severity configurations (Baseline, High, Extreme)
    scenarios = {
        "Moderate (Baseline)": {"base": 100, "sigma": 0.32},
        "High (+30%)": {"base": 130, "sigma": 0.38},
        "Extreme Tail (+65%)": {"base": 165, "sigma": 0.45}
    }
    
    fig = go.Figure()
    
    colors = ['#0ea5e9', '#f59e0b', '#ef4444']
    
    for (name, params), color in zip(scenarios.items(), colors):
        mu = np.log(params["base"]) - (params["sigma"]**2 / 2)
        intensities = np.random.lognormal(mean=mu, sigma=params["sigma"], size=sims)
        
        # Calculate histogram data manually for smooth density curve
        hist, bin_edges = np.histogram(intensities, bins=200, range=(0, 400), density=True)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        fig.add_trace(go.Scatter(
            x=bin_centers, y=hist, mode='lines', fill='tozeroy', 
            name=name, line=dict(color=color, width=2), opacity=0.6
        ))

    fig.update_layout(
        title="Stochastic Hazard Intensity Distributions (Probability Density)",
        xaxis_title="Hazard Intensity (e.g., mph, PGA)",
        yaxis_title="Probability Density",
        legend=dict(x=0.7, y=0.9),
        height=500
    )
    return fig

plot_hazard_distributions().show()

### 2. Vulnerability Engineering (Bounded Logistic Curves)
Vulnerability functions map physical hazard intensity to structural damage. Damage is bounded between $0$ (no damage) and $1$ (total destruction).
Real-world structures exhibit a non-linear response curve: resisting damage initially, then failing rapidly upon exceeding a critical threshold, before plateauing at maximum destruction.

We use a specialized **Bounded Logistic Response Function**:
$$ DR(I) = \min\left( \frac{C_{cap}}{1 + e^{-k(I - I_{mid})}}, C_{cap} \right) $$

Where:
* $I$ = Hazard Intensity
* $I_{mid}$ = The critical inflection point (intensity at which damage accelerates)
* $k$ = The structural sensitivity (steepness of the curve)
* $C_{cap}$ = The asymptotic damage cap (e.g., a steel frame might never reach 100% total collapse from wind alone).

In [ ]:
def calculate_logistic_damage(intensity, mid, k, cap):
    dr = cap / (1.0 + np.exp(-k * (intensity - mid)))
    return np.clip(dr, 0, cap)

def plot_vulnerability_curves():
    intensities = np.linspace(0, 300, 500)
    
    # Real-world parameterized structural models
    vuln_params = {
        "Wood Frame": {"mid": 90.0, "k": 0.045, "cap": 0.97, "color": "#d97706"},
        "Masonry": {"mid": 105.0, "k": 0.038, "cap": 0.92, "color": "#ef4444"},
        "Reinforced Concrete": {"mid": 140.0, "k": 0.032, "cap": 0.85, "color": "#3b82f6"},
        "Engineered Steel": {"mid": 155.0, "k": 0.030, "cap": 0.70, "color": "#10b981"}
    }
    
    fig = go.Figure()
    
    for material, params in vuln_params.items():
        dr = calculate_logistic_damage(intensities, params["mid"], params["k"], params["cap"])
        fig.add_trace(go.Scatter(
            x=intensities, y=dr, mode='lines', name=material,
            line=dict(width=3, color=params["color"])
        ))
        
    # Add inflection point annotations
    fig.add_vline(x=90, line_dash="dot", line_color="rgba(255,255,255,0.2)")
    fig.add_vline(x=155, line_dash="dot", line_color="rgba(255,255,255,0.2)")

    fig.update_layout(
        title="Structural Vulnerability Response Functions",
        xaxis_title="Hazard Intensity",
        yaxis_title="Mean Damage Ratio (MDR)",
        yaxis=dict(tickformat=".0%"),
        height=500
    )
    return fig

plot_vulnerability_curves().show()

### 3. Non-Linear Financial Intelligence (Gross vs Net)
A catastrophic loss is rarely paid purely out-of-pocket by an insurer. Complex layers of **Deductibles** (retentions) and **Policy Limits** (exhaustions) create highly non-linear payouts.

**Ground-Up Loss (GUL)** = $TIV \times MDR$

**Net Insured Loss** = $\min(\max(GUL - Deductible, 0), Limit)$

In [ ]:
def plot_financial_structures():
    # Simulating a single $1M commercial property
    tiv = 1_000_000
    deductible = 50_000   # Insured pays first 50k
    limit = 600_000       # Payout maxes out at 600k (Sub-limit)
    
    damage_ratios = np.linspace(0, 1.0, 500)
    gul = tiv * damage_ratios
    net_loss = np.minimum(np.maximum(gul - deductible, 0), limit)
    
    fig = go.Figure()
    
    # Total potential damage
    fig.add_trace(go.Scatter(x=damage_ratios, y=gul, mode='lines', name='Ground-Up Loss (GUL)', line=dict(color='#94a3b8', dash='dash')))
    
    # What the insurer actually pays
    fig.add_trace(go.Scatter(x=damage_ratios, y=net_loss, mode='lines', fill='tozeroy', name='Net Insured Loss', line=dict(color='#0ea5e9', width=3)))
    
    # Annotations
    fig.add_annotation(x=0.05, y=0, text="Attachment Point (Deductible Met)", showarrow=True, arrowhead=2, ax=50, ay=-50, font=dict(color="#f59e0b"))
    fig.add_annotation(x=0.65, y=600000, text="Exhaustion Point (Limit Hit)", showarrow=True, arrowhead=2, ax=-50, ay=50, font=dict(color="#ef4444"))

    fig.update_layout(
        title="Financial Leakage & Structural Caps (TIV: $1M, Ded: $50k, Lim: $600k)",
        xaxis_title="Damage Ratio (% of Property Destroyed)",
        yaxis_title="Financial Loss ($)",
        xaxis=dict(tickformat=".0%"),
        height=500
    )
    return fig

plot_financial_structures().show()

### 4. Vectorized Portfolio Convergence & The EP Curve
Finally, we run 10,000 simulations against the entire portfolio. Because extreme tail events are exceptionally rare, plotting the frequency requires a logarithmic Exceedance Probability (EP) curve.

* **AAL (Average Annual Loss)**: The expected integral of the loss curve.
* **PML (Probable Maximum Loss)**: Specific percentiles on the tail (e.g., the 1-in-250 year event).

In [ ]:
def plot_complex_ep_curve():
    np.random.seed(99)
    sims = 10000
    
    # Generate synthetic heavily skewed portfolio losses (mimicking our engine)
    # Most years have $0 loss (event didn't happen), some have massive losses.
    occurrences = np.random.rand(sims) < 0.15 # 15% chance of event per year
    losses = np.random.lognormal(mean=14, sigma=1.2, size=sims) * occurrences
    
    sorted_losses = np.sort(losses)[::-1]
    
    # Exceedance Probability (Probability of exceeding loss X)
    probabilities = np.arange(1, sims + 1) / sims
    return_periods = 1.0 / probabilities
    
    # Filter zeros for log plot
    mask = sorted_losses > 0
    rp_filtered = return_periods[mask]
    loss_filtered = sorted_losses[mask]
    
    aal = np.mean(losses)
    pml100 = sorted_losses[int(np.floor(sims / 100)) - 1]
    pml250 = sorted_losses[int(np.floor(sims / 250)) - 1]
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=rp_filtered, y=loss_filtered, mode='lines', 
        name='OEP Curve', line=dict(color='#ef4444', width=3)
    ))
    
    # Annotate AAL (Expected horizontal line)
    fig.add_hline(y=aal, line_dash="dash", line_color="#10b981", annotation_text=f"AAL: ${aal:,.0f}", annotation_position="bottom right")
    
    # Annotate Tail PMLs
    fig.add_vline(x=100, line_dash="dash", line_color="#f59e0b", annotation_text=f"100-Year PML<br>${pml100:,.0f}")
    fig.add_vline(x=250, line_dash="dash", line_color="#ef4444", annotation_text=f"250-Year PML<br>${pml250:,.0f}")

    fig.update_layout(
        title="Occurrence Exceedance Probability (OEP) - Tail Risk Analysis",
        xaxis_title="Return Period (Years)",
        yaxis_title="Aggregate Portfolio Loss ($)",
        xaxis_type="log",
        height=600
    )
    return fig

plot_complex_ep_curve().show()